# Phase 2: Defensive Metrics and Player Evaluation
## 2026 MLB Utility Defender Value Analysis
### Objective
Evaluate the defensive performance of eligible MLB utility players using positional workloads and advanced defensive metrics.
We will investigate whether players who provide meaningful defensive versatility also generate measurable defensive value.

In [106]:
import pandas as pd
import numpy as np
from pathlib import Path
# Define project directories
project_folder = Path.cwd().parent
data_folder = project_folder / "data"
# Load Phase 1 datasets
eligible_df = pd.read_csv(
    data_folder / "2026_eligible_utility_players.csv"
)
position_df = pd.read_csv(
    data_folder / "2026_position_breakdown.csv"
)
fielding_df = pd.read_csv(
    data_folder / "2026_position_fielding.csv"
)
# Verify successful imports
print("Eligible utility defenders:", len(eligible_df))
print("Position breakdown records:", len(position_df))
print("Fielding records:", len(fielding_df))
display(eligible_df.head())

Eligible utility defenders: 107
Position breakdown records: 107
Fielding records: 1326


,player_id,player_name,total_outs,positions_played,total_innings,qualifying_positions,secondary_position,secondary_innings,secondary_share
0,663538,Nico Hoerner,4116,2,1372.000000,2,SS,352.333333,0.256803
1,665487,Fernando Tatis Jr.,4029,2,1343.000000,2,2B,413.000000,0.307520
2,686797,Brooks Lee,4019,2,1339.666667,2,SS,400.000000,0.298582
3,666182,Bo Bichette,4008,3,1336.000000,2,SS,391.333333,0.292914
4,664761,Alec Bohm,3997,2,1332.333333,2,1B,424.666667,0.318739


## Step 2: Inspect Available Defensive Statistics
Examine the existing MLB fielding dataset to identify available defensive statistics and determine which additional metrics must be collected.

In [107]:
# Inspect available columns
print("Available fielding columns:")
print(fielding_df.columns.tolist())
# Examine the first five records
display(fielding_df.head())
# Check column data types
print("\nData types:")
print(fielding_df.dtypes)

Available fielding columns:
['player_id', 'player_name', 'position', 'innings', 'games', 'innings_outs', 'innings_decimal']


,player_id,player_name,position,innings,games,innings_outs,innings_decimal
0,682928,CJ Abrams,2B,103.0,14,309,103.0
1,682668,Luisangel Acuña,2B,1.0,1,3,1.0
2,682668,Luisangel Acuña,LF,3.0,1,9,3.0
3,666176,Jo Adell,LF,96.0,16,288,96.0
4,666176,Jo Adell,CF,10.0,3,30,10.0



Data types:
player_id            int64
player_name         object
position            object
innings            float64
games                int64
innings_outs         int64
innings_decimal    float64
dtype: object


## Step 3: Collect Statcast Outs Above Average
Outs Above Average (OAA) measures defensive performance relative to expected outcomes.
We will investigate the availability of 2026 Statcast OAA data for eligible utility defenders.
OAA will be evaluated alongside positional workload and versatility rather than treated as a complete measure of defensive value.

In [108]:
# Load 2026 Statcast Outs Above Average data
oaa_df = pd.read_csv(
    data_folder / "outs_above_average.csv"
)
print("OAA records:", len(oaa_df))
print("Unique players:", oaa_df["player_id"].nunique())
display(oaa_df.head(10))

OAA records: 273
Unique players: 273


,"last_name, first_name",player_id,display_team_name,year,primary_pos_formatted,fielding_runs_prevented,outs_above_average,outs_above_average_infront,outs_above_average_lateral_toward3bline,outs_above_average_lateral_toward1bline,outs_above_average_behind,outs_above_average_rhh,outs_above_average_lhh,actual_success_rate_formatted,adj_estimated_success_rate_formatted,diff_success_rate_formatted
0,"Abrams, CJ",682928,Nationals,2026,SS,-13,-17,-15,0,-2,1,-15,-3,72%,75%,-3%
1,"Abreu, Wilyer",677800,Red Sox,2026,RF,2,2,1,0,1,0,2,0,91%,90%,1%
2,"Acuña Jr., Ronald",660670,Braves,2026,RF,-9,-10,-1,-4,-3,-1,-4,-6,85%,89%,-4%
3,"Acuña, Luisangel",682668,White Sox,2026,SS,-1,0,-1,-1,2,-1,-1,0,79%,79%,0%
4,"Adames, Willy",642715,Giants,2026,SS,-13,-17,-8,0,-8,-1,-4,-13,75%,79%,-4%
5,"Adell, Jo",666176,---,2026,RF,-6,-6,0,-6,0,0,-1,-6,84%,87%,-3%
6,"Albies, Ozzie",645277,Braves,2026,2B,0,1,2,-4,3,-1,-3,4,74%,74%,0%
7,"Alexander, Blaze",677942,Orioles,2026,3B,-4,-5,-5,-1,2,-1,-2,-2,72%,74%,-2%
8,"Allen, Nick",669397,Astros,2026,SS,-2,-2,-1,0,0,-1,-5,3,77%,78%,-1%
9,"Alonso, Pete",624413,Orioles,2026,1B,0,0,-1,-2,2,1,0,0,76%,76%,0%


In [109]:
# Match eligible utility defenders with Statcast OAA data
matched_df = eligible_df.merge(
    oaa_df,
    on="player_id",
    how="left",
    indicator=True
)
# Check matching results
print("Eligible utility defenders:", len(eligible_df))
print(
    "Players matched with OAA:",
    (matched_df["_merge"] == "both").sum()
)
print(
    "Players without OAA:",
    (matched_df["_merge"] == "left_only").sum()
)
# Display unmatched players
unmatched_df = matched_df[
    matched_df["_merge"] == "left_only"
]
display(
    unmatched_df[["player_id", "player_name"]]
)

Eligible utility defenders: 107
Players matched with OAA: 89
Players without OAA: 18


,player_id,player_name
70,666126,Carlos Cortes
76,660821,Jesús Sánchez
84,621438,Tyrone Taylor
86,665052,Griffin Conine
88,656537,Derek Hill
90,642201,Eli White
91,800325,Luis Lara
92,678489,Jorge Barrosa
96,695720,Tommy White
97,669743,Alex Call


In [110]:
# Step 13: Inspect available Statcast metrics
print("Available OAA columns:")
for column in oaa_df.columns:
    print(column)
print("\nMissing values:")
print(oaa_df.isna().sum().to_string())
print("\nDataset dimensions:")
print(oaa_df.shape)

Available OAA columns:
last_name, first_name
player_id
display_team_name
year
primary_pos_formatted
fielding_runs_prevented
outs_above_average
outs_above_average_infront
outs_above_average_lateral_toward3bline
outs_above_average_lateral_toward1bline
outs_above_average_behind
outs_above_average_rhh
outs_above_average_lhh
actual_success_rate_formatted
adj_estimated_success_rate_formatted
diff_success_rate_formatted

Missing values:
last_name, first_name                      0
player_id                                  0
display_team_name                          0
year                                       0
primary_pos_formatted                      0
fielding_runs_prevented                    0
outs_above_average                         0
outs_above_average_infront                 0
outs_above_average_lateral_toward3bline    0
outs_above_average_lateral_toward1bline    0
outs_above_average_behind                  0
outs_above_average_rhh                     0
outs_above_average_lhh    

In [111]:
# Step 14: Merge eligible defenders with Statcast metrics
utility_metrics_df = eligible_df.merge(
    oaa_df,
    on="player_id",
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("", "_statcast")
)
print("Total utility defenders:", len(utility_metrics_df))
print("\nMerge results:")
print(utility_metrics_df["_merge"].value_counts())

print("\nOAA summary:")
print(
    utility_metrics_df["outs_above_average"].describe()
)
display(
    utility_metrics_df[
        [
            "player_name",
            "total_innings",
            "qualifying_positions",
            "secondary_share",
            "outs_above_average",
            "fielding_runs_prevented"
        ]
    ].head(10)
)

Total utility defenders: 107

Merge results:
_merge
both          89
left_only     18
right_only     0
Name: count, dtype: int64

OAA summary:
count    89.000000
mean     -0.370787
std       6.183523
min     -13.000000
25%      -4.000000
50%       0.000000
75%       4.000000
max      17.000000
Name: outs_above_average, dtype: float64


,player_name,total_innings,qualifying_positions,secondary_share,outs_above_average,fielding_runs_prevented
0,Nico Hoerner,1372.000000,2,0.256803,11.0,8.0
1,Fernando Tatis Jr.,1343.000000,2,0.307520,4.0,4.0
2,Brooks Lee,1339.666667,2,0.298582,0.0,0.0
3,Bo Bichette,1336.000000,2,0.292914,-1.0,-1.0
4,Alec Bohm,1332.333333,2,0.318739,-11.0,-8.0
5,Mauricio Dubón,1325.000000,3,0.344654,6.0,5.0
6,Bryson Stott,1321.000000,2,0.278325,9.0,7.0
7,Kevin McGonigle,1319.666667,2,0.488002,-3.0,-3.0
8,Sal Stewart,1311.666667,2,0.233799,-2.0,-1.0
9,Luke Keaschall,1298.666667,3,0.347793,-12.0,-10.0


In [112]:
# Step 15: Calculate workload-adjusted OAA
utility_metrics_df["oaa_per_1000_innings"] = (
    utility_metrics_df["outs_above_average"]
    / utility_metrics_df["total_innings"]
    * 1000
)
print("OAA per 1,000 innings summary:")
print(
    utility_metrics_df["oaa_per_1000_innings"].describe()
)
print("\nTop 10 players by OAA per 1,000 innings:")
top_oaa_rate = (
    utility_metrics_df
    .dropna(subset=["oaa_per_1000_innings"])
    .sort_values("oaa_per_1000_innings", ascending=False)
    [
        [
            "player_name",
            "total_innings",
            "outs_above_average",
            "oaa_per_1000_innings"
        ]
    ]
    .head(10)
)
display(top_oaa_rate.round(2))

OAA per 1,000 innings summary:
count    89.000000
mean     -0.429619
std       7.251193
min     -15.693112
25%      -4.862237
50%       0.000000
75%       3.745318
max      18.113208
Name: oaa_per_1000_innings, dtype: float64

Top 10 players by OAA per 1,000 innings:


,player_name,total_innings,outs_above_average,oaa_per_1000_innings
94,Isiah Kiner-Falefa,441.67,8.0,18.11
41,Jonny DeLuca,787.00,14.0,17.79
28,Joey Ortiz,973.00,15.0,15.42
11,Ezequiel Duran,1282.33,17.0,13.26
82,Brice Matthews,531.00,6.0,11.30
66,Daniel Schneemann,623.67,7.0,11.22
98,Sung-Mun Song,389.33,4.0,10.27
13,Colson Montgomery,1243.67,12.0,9.65
30,Zach McKinstry,955.33,8.0,8.37
0,Nico Hoerner,1372.00,11.0,8.02


In [113]:
# Step 16: Calculate positional versatility
import numpy as np
def versatility_entropy(shares):
    shares = np.asarray(shares, dtype=float)
    shares = shares[shares > 0]
    if len(shares) <= 1:
        return 0.0
    entropy = -np.sum(shares * np.log(shares))
    return entropy / np.log(len(shares))
# Inspect available position breakdown columns
print("Position breakdown columns:")
print(position_df.columns.tolist())
display(position_df.head())

Position breakdown columns:
['player_id', 'total_innings', 'player_name', '1B', '2B', '3B', 'CF', 'LF', 'RF', 'SS', 'calculated_total', 'innings_difference']


,player_id,total_innings,player_name,1B,2B,3B,CF,LF,RF,SS,calculated_total,innings_difference
0,663538,1372.000000,Nico Hoerner,0.000000,1019.666667,0.000000,0.0,0.0,0.0,352.333333,1372.000000,0.000000e+00
1,665487,1343.000000,Fernando Tatis Jr.,0.000000,413.000000,0.000000,0.0,0.0,930.0,0.000000,1343.000000,0.000000e+00
2,686797,1339.666667,Brooks Lee,0.000000,0.000000,939.666667,0.0,0.0,0.0,400.000000,1339.666667,2.273737e-13
3,666182,1336.000000,Bo Bichette,0.000000,1.000000,943.666667,0.0,0.0,0.0,391.333333,1336.000000,0.000000e+00
4,664761,1332.333333,Alec Bohm,424.666667,0.000000,907.666667,0.0,0.0,0.0,0.000000,1332.333333,0.000000e+00


In [114]:
# Step 17: Calculate positional versatility scores
position_cols = ["1B", "2B", "3B", "SS", "LF", "CF", "RF"]
# Calculate each position's share of total defensive innings
position_shares = position_df[position_cols].div(
    position_df["total_innings"],
    axis=0
)
# Calculate normalized Shannon entropy
position_df["versatility_score"] = position_shares.apply(
    lambda row: versatility_entropy(row.values),
    axis=1
)
# Merge versatility into our combined defensive dataset
utility_metrics_df = utility_metrics_df.merge(
    position_df[["player_id", "versatility_score"]],
    on="player_id",
    how="left",
    validate="one_to_one"
)
print("Versatility score summary:")
print(utility_metrics_df["versatility_score"].describe())
print("\nTop 15 most balanced utility defenders:")
top_versatile = (
    utility_metrics_df
    .sort_values("versatility_score", ascending=False)
    [
        [
            "player_name",
            "total_innings",
            "qualifying_positions",
            "secondary_share",
            "versatility_score",
            "outs_above_average"
        ]
    ]
    .head(15)
)
display(top_versatile.round(3))

Versatility score summary:
count    107.000000
mean       0.772345
std        0.146821
min        0.398750
25%        0.671660
50%        0.793728
75%        0.878812
max        1.000000
Name: versatility_score, dtype: float64

Top 15 most balanced utility defenders:


,player_name,total_innings,qualifying_positions,secondary_share,versatility_score,outs_above_average
54,Ryan O'Hearn,670.667,2,0.500,1.000,-3.0
70,Carlos Cortes,584.333,2,0.500,1.000,NaN
12,Steven Kwan,1263.667,2,0.496,1.000,6.0
47,Jorbit Vivas,747.333,2,0.493,1.000,4.0
76,Jesús Sánchez,559.000,2,0.491,1.000,NaN
7,Kevin McGonigle,1319.667,2,0.488,1.000,-3.0
96,Tommy White,414.000,2,0.437,0.989,NaN
99,Jordan Beck,387.000,2,0.432,0.987,NaN
46,Myles Straw,760.667,3,0.384,0.971,4.0
98,Sung-Mun Song,389.333,2,0.298,0.966,4.0


In [115]:
# Step 18: Calculate meaningful positional breadth
position_cols = ["1B", "2B", "3B", "SS", "LF", "CF", "RF"]
# Count positions with at least 100 defensive innings
position_df["meaningful_positions"] = (
    position_df[position_cols] >= 100
).sum(axis=1)
# Merge into utility defender dataset
utility_metrics_df = utility_metrics_df.merge(
    position_df[["player_id", "meaningful_positions"]],
    on="player_id",
    how="left",
    validate="one_to_one"
)
print("Meaningful positions distribution:")
print(
    utility_metrics_df["meaningful_positions"]
    .value_counts()
    .sort_index()
)
print("\nPlayers covering the most positions:")
display(
    utility_metrics_df.sort_values(
        ["meaningful_positions", "total_innings"],
        ascending=[False, False]
    )[
        [
            "player_name",
            "total_innings",
            "meaningful_positions",
            "versatility_score",
            "outs_above_average"
        ]
    ].head(15).round(3)
)

Meaningful positions distribution:
meaningful_positions
2    82
3    24
4     1
Name: count, dtype: int64

Players covering the most positions:


,player_name,total_innings,meaningful_positions,versatility_score,outs_above_average
40,Spencer Steer,812.000,4,0.803,-2.0
5,Mauricio Dubón,1325.000,3,0.713,6.0
9,Luke Keaschall,1298.667,3,0.921,-12.0
11,Ezequiel Duran,1282.333,3,0.842,17.0
14,Kody Clemens,1212.333,3,0.794,1.0
18,Jake McCarthy,1111.333,3,0.862,1.0
19,Royce Lewis,1097.333,3,0.799,-4.0
21,José Caballero,1072.667,3,0.742,2.0
26,Jake Mangum,986.333,3,0.921,-1.0
29,Willi Castro,960.667,3,0.706,-6.0


##  Evaluate Defensive Positional Responsibilities
Examine the distribution of defensive innings across positions
to understand each player's defensive responsibilities.
This analysis distinguishes positional workload from actual
defensive performance. Positional difficulty will be considered
separately when developing the final utility defender value model.

In [116]:
# Step 19: Calculate defensive position workload shares
demanding_positions = ["SS", "2B", "3B", "CF"]
position_df["demanding_innings"] = (
    position_df[demanding_positions].sum(axis=1)
)
position_df["demanding_position_share"] = (
    position_df["demanding_innings"]
    / position_df["total_innings"]
)
# Merge workload share into our evaluation dataset
utility_metrics_df = utility_metrics_df.merge(
    position_df[["player_id", "demanding_position_share"]],
    on="player_id",
    how="left",
    validate="one_to_one"
)
print("Demanding position share summary:")
print(
    utility_metrics_df["demanding_position_share"]
    .describe()
)
print("\nTop 15 players by demanding position workload:")
display(
    utility_metrics_df.sort_values(
        ["demanding_position_share", "total_innings"],
        ascending=[False, False]
    )[
        [
            "player_name",
            "total_innings",
            "meaningful_positions",
            "demanding_position_share",
            "outs_above_average"
        ]
    ].head(15).round(3)
)

Demanding position share summary:
count    107.000000
mean       0.601153
std        0.360042
min        0.000000
25%        0.315454
50%        0.652207
75%        0.991909
max        1.000000
Name: demanding_position_share, dtype: float64

Top 15 players by demanding position workload:


,player_name,total_innings,meaningful_positions,demanding_position_share,outs_above_average
16,Nick Gonzales,1152.333,2,1.0,-11.0
44,Hao-Yu Lee,764.667,2,1.0,-12.0
77,Alika Williams,547.667,2,1.0,2.0
0,Nico Hoerner,1372.000,2,1.0,11.0
3,Bo Bichette,1336.000,2,1.0,-1.0
6,Bryson Stott,1321.000,2,1.0,9.0
13,Colson Montgomery,1243.667,2,1.0,12.0
15,Nasim Nuñez,1161.000,2,1.0,4.0
17,Matt McLain,1114.333,2,1.0,4.0
27,J.P. Crawford,974.333,2,1.0,-13.0


In [117]:
# Step 20: Validate metrics before modeling
metrics = [
    "total_innings",
    "outs_above_average",
    "oaa_per_1000_innings",
    "versatility_score",
    "meaningful_positions",
    "demanding_position_share"
]
# Check missing values
print("Missing values:")
print(utility_metrics_df[metrics].isna().sum())
# Summary statistics
print("\nMetric summary:")
display(utility_metrics_df[metrics].describe().round(3))
# Correlations using available observations
print("\nCorrelation matrix:")
correlation_matrix = utility_metrics_df[metrics].corr()
display(correlation_matrix.round(3))

Missing values:
total_innings                0
outs_above_average          18
oaa_per_1000_innings        18
versatility_score            0
meaningful_positions         0
demanding_position_share     0
dtype: int64

Metric summary:


,total_innings,outs_above_average,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share
count,107.000,89.000,89.000,107.000,107.000,107.000
mean,770.408,-0.371,-0.430,0.772,2.243,0.601
std,302.020,6.184,7.251,0.147,0.452,0.360
min,300.000,-13.000,-15.693,0.399,2.000,0.000
25%,539.667,-4.000,-4.862,0.672,2.000,0.315
50%,675.333,0.000,0.000,0.794,2.000,0.652
75%,980.333,4.000,3.745,0.879,2.000,0.992
max,1372.000,17.000,18.113,1.000,4.000,1.000



Correlation matrix:


,total_innings,outs_above_average,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share
total_innings,1.000,0.039,-0.005,-0.025,0.176,0.065
outs_above_average,0.039,1.000,0.948,-0.059,-0.000,0.216
oaa_per_1000_innings,-0.005,0.948,1.000,-0.058,-0.005,0.230
versatility_score,-0.025,-0.059,-0.058,1.000,0.249,-0.378
meaningful_positions,0.176,-0.000,-0.005,0.249,1.000,-0.231
demanding_position_share,0.065,0.216,0.230,-0.378,-0.231,1.000


In [118]:
# Step 21: Establish final evaluation sample
# Keep original dataset unchanged
evaluation_df = utility_metrics_df.copy()
# Players with available defensive performance data
ranked_df = evaluation_df.dropna(
    subset=["outs_above_average", "oaa_per_1000_innings"]
).copy()
# Players without OAA
unranked_df = evaluation_df[
    evaluation_df["outs_above_average"].isna()
].copy()
print("Total eligible defenders:", len(evaluation_df))
print("Players eligible for composite ranking:", len(ranked_df))
print("Players excluded due to missing OAA:", len(unranked_df))
print("\nRanked sample summary:")
display(
    ranked_df[
        [
            "player_name",
            "total_innings",
            "outs_above_average",
            "oaa_per_1000_innings",
            "versatility_score",
            "meaningful_positions",
            "demanding_position_share"
        ]
    ].head(10).round(3)
)

Total eligible defenders: 107
Players eligible for composite ranking: 89
Players excluded due to missing OAA: 18

Ranked sample summary:


,player_name,total_innings,outs_above_average,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share
0,Nico Hoerner,1372.000,11.0,8.017,0.822,2,1.000
1,Fernando Tatis Jr.,1343.000,4.0,2.978,0.890,2,0.308
2,Brooks Lee,1339.667,0.0,0.000,0.880,2,1.000
3,Bo Bichette,1336.000,-1.0,-0.749,0.556,2,1.000
4,Alec Bohm,1332.333,-11.0,-8.256,0.903,2,0.681
5,Mauricio Dubón,1325.000,6.0,4.528,0.713,3,0.499
6,Bryson Stott,1321.000,9.0,6.813,0.853,2,1.000
7,Kevin McGonigle,1319.667,-3.0,-2.273,1.000,2,1.000
8,Sal Stewart,1311.667,-2.0,-1.525,0.647,2,0.277
9,Luke Keaschall,1298.667,-12.0,-9.240,0.921,3,0.652


In [119]:
# Step 22: Validate secondary-position playing time
positions = ["1B", "2B", "3B", "SS", "LF", "CF", "RF"]
# Create a separate position analysis dataset
position_analysis = position_df.copy()
# Calculate innings at second-most-played position
position_analysis["second_position_innings"] = (
    position_analysis[positions]
    .apply(lambda row: sorted(row, reverse=True)[1], axis=1)
)
# Calculate secondary-position share
position_analysis["second_position_share"] = (
    position_analysis["second_position_innings"]
    / position_analysis["total_innings"]
)
print("Secondary-position innings summary:")
display(
    position_analysis[
        ["second_position_innings", "second_position_share"]
    ].describe().round(3)
)
print("\nPlayers with the fewest secondary-position innings:")
display(
    position_analysis[
        [
            "player_name",
            "total_innings",
            "second_position_innings",
            "second_position_share"
        ]
    ]
    .sort_values("second_position_innings")
    .head(15)
    .round(3)
)

Secondary-position innings summary:


,second_position_innings,second_position_share
count,107.000,107.000
mean,241.807,0.317
std,109.969,0.078
min,100.000,0.201
25%,155.167,0.261
50%,219.333,0.303
75%,306.333,0.357
max,644.000,0.500



Players with the fewest secondary-position innings:


,player_name,total_innings,second_position_innings,second_position_share
92,Jorge Barrosa,446.333,100.000,0.224
100,Darell Hernaiz,375.000,102.000,0.272
106,Angel Genao,300.000,107.000,0.357
97,Alex Call,399.333,107.000,0.268
104,Hyeseong Kim,317.000,108.000,0.341
94,Isiah Kiner-Falefa,441.667,109.000,0.247
77,Alika Williams,547.667,110.000,0.201
98,Sung-Mun Song,389.333,116.000,0.298
80,Colt Keith,537.333,117.000,0.218
91,Luis Lara,453.000,118.000,0.260


In [120]:
# Step 23: Investigate missing OAA
evaluation_df["oaa_available"] = (
    evaluation_df["outs_above_average"].notna()
)
print("Playing time by OAA availability:")
display(
    evaluation_df.groupby("oaa_available")[
        ["total_innings", "meaningful_positions",
         "versatility_score"]
    ].agg(["count", "mean", "median", "min", "max"]).round(2)
)
print("\nPlayers without OAA:")
display(
    unranked_df[
        ["player_name", "total_innings",
         "meaningful_positions", "versatility_score"]
    ].sort_values("total_innings", ascending=False)
    .round(3)
)

Playing time by OAA availability:


total_innings                                   \
                      count    mean  median     min      max   
oaa_available                                                  
False                    18  424.04  406.67  300.00   584.33   
True                     89  840.46  764.67  389.33  1372.00   

              meaningful_positions                      versatility_score  \
                             count  mean median min max             count   
oaa_available                                                               
False                           18  2.11    2.0   2   3                18   
True                            89  2.27    2.0   2   4                89   

                                      
               mean median  min  max  
oaa_available                         
False          0.83   0.87  0.6  1.0  
True           0.76   0.79  0.4  1.0


Players without OAA:


,player_name,total_innings,meaningful_positions,versatility_score
70,Carlos Cortes,584.333,2,1.000
76,Jesús Sánchez,559.000,2,1.000
84,Tyrone Taylor,527.333,3,0.950
86,Griffin Conine,522.333,2,0.872
88,Derek Hill,508.333,2,0.642
90,Eli White,457.667,3,0.779
91,Luis Lara,453.000,2,0.598
92,Jorge Barrosa,446.333,2,0.899
96,Tommy White,414.000,2,0.989
97,Alex Call,399.333,2,0.848


In [121]:
# Step 24: Construct composite utility defender score
from sklearn.preprocessing import MinMaxScaler
scoring_df = ranked_df.copy()
weights = {
    "oaa_per_1000_innings": 0.45,
    "versatility_score": 0.25,
    "meaningful_positions": 0.15,
    "demanding_position_share": 0.10,
    "total_innings": 0.05
}
scaler = MinMaxScaler()
metric_columns = list(weights.keys())
normalized = scaler.fit_transform(
    scoring_df[metric_columns]
)
normalized_df = pd.DataFrame(
    normalized,
    columns=metric_columns,
    index=scoring_df.index
)
# Weighted composite score (0–100)
scoring_df["utility_defender_score"] = sum(
    normalized_df[metric] * weight * 100
    for metric, weight in weights.items()
)
# Rank players
scoring_df["rank"] = (
    scoring_df["utility_defender_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)
top_20 = scoring_df.sort_values(
    "utility_defender_score", ascending=False
).head(20)
display(
    top_20[
        [
            "rank",
            "player_name",
            "utility_defender_score",
            "outs_above_average",
            "oaa_per_1000_innings",
            "versatility_score",
            "meaningful_positions",
            "demanding_position_share",
            "total_innings"
        ]
    ].round(3)
)

,rank,player_name,utility_defender_score,outs_above_average,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share,total_innings
11,1,Ezequiel Duran,77.573,17.0,13.257,0.842,3,0.857,1282.333
98,2,Sung-Mun Song,68.160,4.0,10.274,0.966,2,1.000,389.333
82,3,Brice Matthews,67.611,6.0,11.299,0.787,3,0.731,531.000
94,4,Isiah Kiner-Falefa,65.588,8.0,18.113,0.649,2,0.993,441.667
46,5,Myles Straw,65.035,4.0,5.259,0.971,3,0.397,760.667
47,6,Jorbit Vivas,64.829,4.0,5.352,1.000,2,1.000,747.333
0,7,Nico Hoerner,64.155,11.0,8.017,0.822,2,1.000,1372.000
41,8,Jonny DeLuca,64.025,14.0,17.789,0.719,2,0.412,787.000
6,9,Bryson Stott,63.593,9.0,6.813,0.853,2,1.000,1321.000
30,10,Zach McKinstry,62.763,8.0,8.374,0.756,3,0.550,955.333


In [122]:
# Step 25: Sensitivity analysis
weight_scenarios = {
    "balanced": {
        "oaa_per_1000_innings": 0.45,
        "versatility_score": 0.25,
        "meaningful_positions": 0.15,
        "demanding_position_share": 0.10,
        "total_innings": 0.05
    },
    "defense_focused": {
        "oaa_per_1000_innings": 0.60,
        "versatility_score": 0.15,
        "meaningful_positions": 0.10,
        "demanding_position_share": 0.10,
        "total_innings": 0.05
    },
    "versatility_focused": {
        "oaa_per_1000_innings": 0.30,
        "versatility_score": 0.35,
        "meaningful_positions": 0.20,
        "demanding_position_share": 0.10,
        "total_innings": 0.05
    }
}
for scenario, scenario_weights in weight_scenarios.items():
    scoring_df[f"{scenario}_score"] = sum(
        normalized_df[metric] * weight * 100
        for metric, weight in scenario_weights.items()
    )
    scoring_df[f"{scenario}_rank"] = (
        scoring_df[f"{scenario}_score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )
comparison = scoring_df[
    [
        "player_name",
        "balanced_rank",
        "defense_focused_rank",
        "versatility_focused_rank"
    ]
].sort_values("balanced_rank")

display(comparison.head(20))

,player_name,balanced_rank,defense_focused_rank,versatility_focused_rank
11,Ezequiel Duran,1,1,1
98,Sung-Mun Song,2,6,3
82,Brice Matthews,3,5,5
94,Isiah Kiner-Falefa,4,2,26
46,Myles Straw,5,14,2
47,Jorbit Vivas,6,12,4
0,Nico Hoerner,7,8,11
41,Jonny DeLuca,8,3,27
6,Bryson Stott,9,10,9
30,Zach McKinstry,10,11,12


In [123]:
import numpy as np
rank_columns = [
    "balanced_rank",
    "defense_focused_rank",
    "versatility_focused_rank"
]
# Calculate ranking stability
scoring_df["rank_std"] = scoring_df[rank_columns].std(
    axis=1
)
scoring_df["rank_range"] = (
    scoring_df[rank_columns].max(axis=1)
    - scoring_df[rank_columns].min(axis=1)
)
# Display top 20 balanced-model players
stability_results = scoring_df[
    [
        "player_name",
        "balanced_rank",
        "defense_focused_rank",
        "versatility_focused_rank",
        "rank_std",
        "rank_range"
    ]
].sort_values("balanced_rank")
display(stability_results.head(20).round(2))
# Overall ranking stability
print("Average rank standard deviation:",
      round(scoring_df["rank_std"].mean(), 2))
print("Median rank range:",
      scoring_df["rank_range"].median())

,player_name,balanced_rank,defense_focused_rank,versatility_focused_rank,rank_std,rank_range
11,Ezequiel Duran,1,1,1,0.00,0
98,Sung-Mun Song,2,6,3,2.08,4
82,Brice Matthews,3,5,5,1.15,2
94,Isiah Kiner-Falefa,4,2,26,13.32,24
46,Myles Straw,5,14,2,6.24,12
47,Jorbit Vivas,6,12,4,4.16,8
0,Nico Hoerner,7,8,11,2.08,4
41,Jonny DeLuca,8,3,27,12.66,24
6,Bryson Stott,9,10,9,0.58,1
30,Zach McKinstry,10,11,12,1.00,2


Average rank standard deviation: 7.0
Median rank range: 12.0


In [124]:
# Step 27: Evaluate playing-time bias
import pandas as pd
# Divide players into playing-time groups
scoring_df["innings_group"] = pd.cut(
    scoring_df["total_innings"],
    bins=[0, 500, 800, 1100, float("inf")],
    labels=[
        "Under 500",
        "500-799",
        "800-1099",
        "1100+"
    ],
    right=False
)
# Compare rankings across playing-time groups
playing_time_summary = (
    scoring_df.groupby(
        "innings_group", observed=True
    )
    .agg(
        players=("player_name", "count"),
        avg_score=("balanced_score", "mean"),
        avg_rank=("balanced_rank", "mean"),
        avg_oaa_rate=("oaa_per_1000_innings", "mean")
    )
    .round(2)
)
display(playing_time_summary)
# Inspect the top 20 players
top_20_playing_time = (
    scoring_df.sort_values("balanced_rank")
    [
        [
            "player_name",
            "balanced_rank",
            "total_innings",
            "outs_above_average",
            "oaa_per_1000_innings",
            "innings_group"
        ]
    ]
    .head(20)
)

display(top_20_playing_time.round(2))
# Check relationship between innings and ranking
correlation = scoring_df[
    ["total_innings", "balanced_rank"]
].corr().iloc[0, 1]
print(
    "Correlation between innings and ranking:",
    round(correlation, 3)
)

,players,avg_score,avg_rank,avg_oaa_rate
innings_group,,,,
Under 500,5,49.91,37.20,3.83
500-799,43,44.94,46.86,-0.48
800-1099,22,41.60,54.59,-2.59
1100+,19,52.76,31.74,1.06


,player_name,balanced_rank,total_innings,outs_above_average,oaa_per_1000_innings,innings_group
11,Ezequiel Duran,1,1282.33,17.0,13.26,1100+
98,Sung-Mun Song,2,389.33,4.0,10.27,Under 500
82,Brice Matthews,3,531.00,6.0,11.30,500-799
94,Isiah Kiner-Falefa,4,441.67,8.0,18.11,Under 500
46,Myles Straw,5,760.67,4.0,5.26,500-799
47,Jorbit Vivas,6,747.33,4.0,5.35,500-799
0,Nico Hoerner,7,1372.00,11.0,8.02,1100+
41,Jonny DeLuca,8,787.00,14.0,17.79,500-799
6,Bryson Stott,9,1321.00,9.0,6.81,1100+
30,Zach McKinstry,10,955.33,8.0,8.37,800-1099


Correlation between innings and ranking: -0.159


In [125]:
# Step 28: Test playing-time reliability adjustment
import numpy as np
# Set an initial reliability parameter
reliability_k = 500
# Calculate reliability factor
scoring_df["reliability_factor"] = (
    scoring_df["total_innings"] /
    (scoring_df["total_innings"] + reliability_k)
)
# Adjust OAA rate
scoring_df["adjusted_oaa_rate"] = (
    scoring_df["oaa_per_1000_innings"] *
    scoring_df["reliability_factor"]
)
# Compare original and adjusted rates
comparison = scoring_df[
    [
        "player_name",
        "balanced_rank",
        "total_innings",
        "oaa_per_1000_innings",
        "reliability_factor",
        "adjusted_oaa_rate"
    ]
].sort_values("balanced_rank")

display(comparison.head(20).round(3))

,player_name,balanced_rank,total_innings,oaa_per_1000_innings,reliability_factor,adjusted_oaa_rate
11,Ezequiel Duran,1,1282.333,13.257,0.719,9.538
98,Sung-Mun Song,2,389.333,10.274,0.438,4.498
82,Brice Matthews,3,531.000,11.299,0.515,5.820
94,Isiah Kiner-Falefa,4,441.667,18.113,0.469,8.496
46,Myles Straw,5,760.667,5.259,0.603,3.173
47,Jorbit Vivas,6,747.333,5.352,0.599,3.207
0,Nico Hoerner,7,1372.000,8.017,0.733,5.876
41,Jonny DeLuca,8,787.000,17.789,0.611,10.878
6,Bryson Stott,9,1321.000,6.813,0.725,4.942
30,Zach McKinstry,10,955.333,8.374,0.656,5.497


In [126]:
# Step 29: Compare original and adjusted rankings
# Create an adjusted copy of normalized metrics
adjusted_normalized = normalized_df.copy()
# Normalize adjusted OAA using min-max scaling
oaa_min = scoring_df["adjusted_oaa_rate"].min()
oaa_max = scoring_df["adjusted_oaa_rate"].max()
adjusted_normalized["oaa_per_1000_innings"] = (
    (scoring_df["adjusted_oaa_rate"] - oaa_min)
    / (oaa_max - oaa_min)
)
# Apply balanced-model weights
scoring_df["adjusted_composite_score"] = sum(
    adjusted_normalized[metric] * weight * 100
    for metric, weight in weight_scenarios["balanced"].items()
)
# Recalculate rankings
scoring_df["adjusted_rank"] = (
    scoring_df["adjusted_composite_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)
# Positive change = improved ranking
scoring_df["rank_change"] = (
    scoring_df["balanced_rank"]
    - scoring_df["adjusted_rank"]
)
# Compare results
rank_comparison = scoring_df[
    [
        "player_name",
        "total_innings",
        "balanced_rank",
        "adjusted_rank",
        "rank_change",
        "adjusted_composite_score"
    ]
].sort_values("adjusted_rank")
display(rank_comparison.head(20).round(2))
print(
    "Average absolute ranking change:",
    round(scoring_df["rank_change"].abs().mean(), 2)
)
print(
    "Largest ranking change:",
    scoring_df["rank_change"].abs().max()
)

,player_name,total_innings,balanced_rank,adjusted_rank,rank_change,adjusted_composite_score
11,Ezequiel Duran,1282.33,1,1,0,81.08
0,Nico Hoerner,1372.00,7,2,5,66.54
6,Bryson Stott,1321.00,9,3,6,65.52
82,Brice Matthews,531.00,3,4,-1,65.50
46,Myles Straw,760.67,5,5,0,65.12
13,Colson Montgomery,1243.67,11,6,5,64.99
47,Jorbit Vivas,747.33,6,7,-1,64.87
98,Sung-Mun Song,389.33,2,8,-6,64.50
41,Jonny DeLuca,787.00,8,9,-1,64.46
30,Zach McKinstry,955.33,10,10,0,63.84


Average absolute ranking change: 1.66
Largest ranking change: 9


In [127]:
# Step 30: Audit defensive data consistency
print("OAA data columns:")
print(oaa_df.columns.tolist())
print("\nPosition fielding columns:")
print(position_df.columns.tolist())
print("\nOAA position distribution:")
print(
    oaa_df["primary_pos_formatted"]
    .value_counts(dropna=False)
)
print("\nSample player position breakdowns:")
sample_players = [
    "Ezequiel Durán",
    "Nico Hoerner",
    "Bryson Stott",
    "Mauricio Dubón"
]
display(
    position_df[
        position_df["player_name"].isin(sample_players)
    ].T
)
print("\nOAA data for selected players:")

sample_ids = position_df.loc[
    position_df["player_name"].isin(sample_players),
    "player_id"
]
display(
    oaa_df[
        oaa_df["player_id"].isin(sample_ids)
    ]
)

OAA data columns:
['last_name, first_name', 'player_id', 'display_team_name', 'year', 'primary_pos_formatted', 'fielding_runs_prevented', 'outs_above_average', 'outs_above_average_infront', 'outs_above_average_lateral_toward3bline', 'outs_above_average_lateral_toward1bline', 'outs_above_average_behind', 'outs_above_average_rhh', 'outs_above_average_lhh', 'actual_success_rate_formatted', 'adj_estimated_success_rate_formatted', 'diff_success_rate_formatted']

Position fielding columns:
['player_id', 'total_innings', 'player_name', '1B', '2B', '3B', 'CF', 'LF', 'RF', 'SS', 'calculated_total', 'innings_difference', 'versatility_score', 'meaningful_positions', 'demanding_innings', 'demanding_position_share']

OAA position distribution:
primary_pos_formatted
2B    49
SS    46
3B    43
CF    41
1B    37
LF    31
RF    26
Name: count, dtype: int64

Sample player position breakdowns:


,0,5,6
player_id,663538,643289,681082
total_innings,1372.0,1325.0,1321.0
player_name,Nico Hoerner,Mauricio Dubón,Bryson Stott
1B,0.0,0.0,0.0
2B,1019.666667,34.0,953.333333
3B,0.0,46.0,367.666667
CF,0.0,125.0,0.0
LF,0.0,663.333333,0.0
RF,0.0,0.0,0.0
SS,352.333333,456.666667,0.0



OAA data for selected players:


,"last_name, first_name",player_id,display_team_name,year,primary_pos_formatted,fielding_runs_prevented,outs_above_average,outs_above_average_infront,outs_above_average_lateral_toward3bline,outs_above_average_lateral_toward1bline,outs_above_average_behind,outs_above_average_rhh,outs_above_average_lhh,actual_success_rate_formatted,adj_estimated_success_rate_formatted,diff_success_rate_formatted
63,"Dubón, Mauricio",643289,Braves,2026,SS,5,6,1,3,1,1,3,3,83%,81%,2%
105,"Hoerner, Nico",663538,Cubs,2026,2B,8,11,2,3,4,2,6,6,75%,73%,2%
228,"Stott, Bryson",681082,Phillies,2026,2B,7,9,6,2,1,0,5,4,70%,68%,2%


In [128]:
# Step 31: Validate OAA metrics and innings totals
# Inspect all available OAA columns
print("Statcast OAA columns:")
for col in oaa_df.columns:
    print(col)
# Check whether positional innings sum correctly
position_columns = ["1B", "2B", "3B", "SS", "LF", "CF", "RF"]
audit_df = position_df.copy()
audit_df["position_sum"] = audit_df[position_columns].sum(axis=1)
audit_df["difference"] = (
    audit_df["total_innings"] - audit_df["position_sum"]
)
print("\nInnings consistency summary:")
print(audit_df["difference"].describe())
print("\nPlayers with discrepancies greater than 1 inning:")
display(
    audit_df.loc[
        audit_df["difference"].abs() > 1,
        ["player_name", "total_innings", "position_sum", "difference"]
    ]
)
print("\nStatcast OAA metric summary:")
display(
    oaa_df[
        ["outs_above_average", "fielding_runs_prevented"]
    ].describe()
)

Statcast OAA columns:
last_name, first_name
player_id
display_team_name
year
primary_pos_formatted
fielding_runs_prevented
outs_above_average
outs_above_average_infront
outs_above_average_lateral_toward3bline
outs_above_average_lateral_toward1bline
outs_above_average_behind
outs_above_average_rhh
outs_above_average_lhh
actual_success_rate_formatted
adj_estimated_success_rate_formatted
diff_success_rate_formatted

Innings consistency summary:
count    1.070000e+02
mean     1.646865e-14
std      6.762386e-14
min     -2.273737e-13
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.273737e-13
Name: difference, dtype: float64

Players with discrepancies greater than 1 inning:


,player_name,total_innings,position_sum,difference



Statcast OAA metric summary:


,outs_above_average,fielding_runs_prevented
count,273.000000,273.000000
mean,0.695971,0.523810
std,7.217668,5.815198
min,-17.000000,-13.000000
25%,-4.000000,-3.000000
50%,0.000000,0.000000
75%,4.000000,4.000000
max,26.000000,23.000000


In [129]:
# Step 32: Audit Statcast OAA player records
print("Total OAA records:", len(oaa_df))
print("Unique player IDs:", oaa_df["player_id"].nunique())
print("\nDuplicate player IDs:")
display(
    oaa_df[
        oaa_df.duplicated("player_id", keep=False)
    ]
)
print("\nPrimary position distribution:")
print(
    oaa_df["primary_pos_formatted"].value_counts()
)
print("\nSelected utility defenders:")
selected_ids = [663538, 643289, 681082]
display(
    oaa_df.loc[
        oaa_df["player_id"].isin(selected_ids),
        [
            "last_name, first_name",
            "player_id",
            "primary_pos_formatted",
            "outs_above_average",
            "fielding_runs_prevented"
        ]
    ]
)

Total OAA records: 273
Unique player IDs: 273

Duplicate player IDs:


,"last_name, first_name",player_id,display_team_name,year,primary_pos_formatted,fielding_runs_prevented,outs_above_average,outs_above_average_infront,outs_above_average_lateral_toward3bline,outs_above_average_lateral_toward1bline,outs_above_average_behind,outs_above_average_rhh,outs_above_average_lhh,actual_success_rate_formatted,adj_estimated_success_rate_formatted,diff_success_rate_formatted



Primary position distribution:
primary_pos_formatted
2B    49
SS    46
3B    43
CF    41
1B    37
LF    31
RF    26
Name: count, dtype: int64

Selected utility defenders:


,"last_name, first_name",player_id,primary_pos_formatted,outs_above_average,fielding_runs_prevented
63,"Dubón, Mauricio",643289,SS,6,5
105,"Hoerner, Nico",663538,2B,11,8
228,"Stott, Bryson",681082,2B,9,7


### Step 33: Statcast OAA Data Validation
The 2026 Statcast Outs Above Average dataset was downloaded
from Baseball Savant with the "All Positions" filter selected.
This allows the analysis to evaluate each player's combined
defensive performance across multiple positions.
Validation confirmed:
- 273 unique player records
- No duplicate player IDs
- Consistent positional innings totals
- 89 of 107 eligible utility defenders matched with OAA
Players without available OAA were excluded from the
composite rankings rather than assigned an artificial value.
The final evaluation incorporates:
- Outs Above Average per 1,000 defensive innings
- Positional versatility
- Number of meaningfully played positions
- Defensive positional responsibilities
- Playing-time reliability adjustments
The composite score is a custom analytical measure,
not an official MLB or Statcast statistic.

In [130]:
# Step 34: Export final utility defender rankings
# Use the playing-time-adjusted rankings
final_rankings = ranked_df.copy()
# Check available columns before exporting
print("Available columns:")
print(final_rankings.columns.tolist())

Available columns:
['player_id', 'player_name', 'total_outs', 'positions_played', 'total_innings', 'qualifying_positions', 'secondary_position', 'secondary_innings', 'secondary_share', 'last_name, first_name', 'display_team_name', 'year', 'primary_pos_formatted', 'fielding_runs_prevented', 'outs_above_average', 'outs_above_average_infront', 'outs_above_average_lateral_toward3bline', 'outs_above_average_lateral_toward1bline', 'outs_above_average_behind', 'outs_above_average_rhh', 'outs_above_average_lhh', 'actual_success_rate_formatted', 'adj_estimated_success_rate_formatted', 'diff_success_rate_formatted', '_merge', 'oaa_per_1000_innings', 'versatility_score', 'meaningful_positions', 'demanding_position_share']


In [131]:
# Step 34: Export final utility defender rankings
# Start with the playing-time-adjusted results
final_export = ranked_df.copy()
# Identify available columns
export_columns = [
    "player_id",
    "player_name",
    "total_innings",
    "outs_above_average",
    "oaa_per_1000_innings",
    "versatility_score",
    "meaningful_positions",
    "demanding_position_share",
    "utility_defender_score",
    "balanced_rank",
    "adjusted_rank",
    "adjusted_composite_score"
]
# Keep columns that exist
available_columns = [
    col for col in export_columns
    if col in final_export.columns
]
final_export = final_export[available_columns]
# Sort by adjusted rank
if "adjusted_rank" in final_export.columns:
    final_export = final_export.sort_values("adjusted_rank")
else:
    print("WARNING: adjusted_rank not found. Export is not yet sorted by final ranking.")
# Save results
output_path = data_folder / "2026_utility_defender_rankings.csv"
final_export.to_csv(output_path, index=False)
print("File saved:", output_path)
print("Total players:", len(final_export))
print("Exported columns:", available_columns)

display(final_export.head(15).round(3))

File saved: /Users/cameronboardman/Desktop/MLB-2026-Utility-Defender-Value/data/2026_utility_defender_rankings.csv
Total players: 89
Exported columns: ['player_id', 'player_name', 'total_innings', 'outs_above_average', 'oaa_per_1000_innings', 'versatility_score', 'meaningful_positions', 'demanding_position_share']


,player_id,player_name,total_innings,outs_above_average,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share
0,663538,Nico Hoerner,1372.000,11.0,8.017,0.822,2,1.000
1,665487,Fernando Tatis Jr.,1343.000,4.0,2.978,0.890,2,0.308
2,686797,Brooks Lee,1339.667,0.0,0.000,0.880,2,1.000
3,666182,Bo Bichette,1336.000,-1.0,-0.749,0.556,2,1.000
4,664761,Alec Bohm,1332.333,-11.0,-8.256,0.903,2,0.681
5,643289,Mauricio Dubón,1325.000,6.0,4.528,0.713,3,0.499
6,681082,Bryson Stott,1321.000,9.0,6.813,0.853,2,1.000
7,805808,Kevin McGonigle,1319.667,-3.0,-2.273,1.000,2,1.000
8,701398,Sal Stewart,1311.667,-2.0,-1.525,0.647,2,0.277
9,807712,Luke Keaschall,1298.667,-12.0,-9.240,0.921,3,0.652


In [132]:
# Step 34: Locate the final adjusted ranking columns
print("Ranking-related columns in ranked_df:")
print([
    col for col in ranked_df.columns
    if "rank" in col.lower()
    or "score" in col.lower()
    or "adjust" in col.lower()
])
print("\nRanking-related columns in comparison:")
print([
    col for col in comparison.columns
    if "rank" in col.lower()
    or "score" in col.lower()
    or "adjust" in col.lower()
])
print("\nAvailable ranking variables:")
for name in [
    "ranked_df",
    "comparison",
    "ranked_sample",
    "final_rankings"
]:
    if name in globals():
        print(name, globals()[name].shape)

Ranking-related columns in ranked_df:
['versatility_score']

Ranking-related columns in comparison:
['balanced_rank', 'adjusted_oaa_rate']

Available ranking variables:
ranked_df (89, 29)
comparison (89, 6)
final_rankings (89, 29)


In [133]:
# Step 35: Inspect final ranking variables
print("Columns in comparison:")
print(comparison.columns.tolist())

print("\nColumns in final_rankings:")
print(final_rankings.columns.tolist())

print("\nRanking-related variables in memory:")
for name in globals():
    if any(word in name.lower() for word in
           ["adjust", "ranking", "composite", "score"]):
        if not name.startswith("_"):
            print(name)

Columns in comparison:
['player_name', 'balanced_rank', 'total_innings', 'oaa_per_1000_innings', 'reliability_factor', 'adjusted_oaa_rate']

Columns in final_rankings:
['player_id', 'player_name', 'total_outs', 'positions_played', 'total_innings', 'qualifying_positions', 'secondary_position', 'secondary_innings', 'secondary_share', 'last_name, first_name', 'display_team_name', 'year', 'primary_pos_formatted', 'fielding_runs_prevented', 'outs_above_average', 'outs_above_average_infront', 'outs_above_average_lateral_toward3bline', 'outs_above_average_lateral_toward1bline', 'outs_above_average_behind', 'outs_above_average_rhh', 'outs_above_average_lhh', 'actual_success_rate_formatted', 'adj_estimated_success_rate_formatted', 'diff_success_rate_formatted', '_merge', 'oaa_per_1000_innings', 'versatility_score', 'meaningful_positions', 'demanding_position_share']

Ranking-related variables in memory:
adjusted_normalized
final_rankings


In [134]:
# Step 36: Inspect adjusted ranking data
print("Comparison columns:")
for col in comparison.columns:
    print(col)
print("\nAdjusted normalized data:")
print(type(adjusted_normalized))
if hasattr(adjusted_normalized, "shape"):
    print("Shape:", adjusted_normalized.shape)
print("\nComparison preview:")
display(comparison.head(10))

Comparison columns:
player_name
balanced_rank
total_innings
oaa_per_1000_innings
reliability_factor
adjusted_oaa_rate

Adjusted normalized data:
<class 'pandas.core.frame.DataFrame'>
Shape: (89, 5)

Comparison preview:


,player_name,balanced_rank,total_innings,oaa_per_1000_innings,reliability_factor,adjusted_oaa_rate
11,Ezequiel Duran,1,1282.333333,13.257083,0.719469,9.538059
98,Sung-Mun Song,2,389.333333,10.273973,0.437781,4.497751
82,Brice Matthews,3,531.000000,11.299435,0.515034,5.819593
94,Isiah Kiner-Falefa,4,441.666667,18.113208,0.469027,8.495575
46,Myles Straw,5,760.666667,5.258545,0.603384,3.172924
47,Jorbit Vivas,6,747.333333,5.352364,0.599145,3.206841
0,Nico Hoerner,7,1372.000000,8.017493,0.732906,5.876068
41,Jonny DeLuca,8,787.000000,17.789072,0.611500,10.878011
6,Bryson Stott,9,1321.000000,6.813020,0.725426,4.942339
30,Zach McKinstry,10,955.333333,8.374040,0.656436,5.497022


In [135]:
# Step 37: Inspect adjusted scoring components
print("Adjusted normalized columns:")
print(adjusted_normalized.columns.tolist())

print("\nAdjusted normalized preview:")
display(adjusted_normalized.head(10).round(3))

print("\nOriginal ranking columns:")
print(ranked_df.columns.tolist())


print("\nOriginal composite score preview:")

if "utility_defender_score" in ranked_df.columns:
    display(
        ranked_df[
            ["player_name", "utility_defender_score"]
        ].head(10).round(3)
    )
elif "utility_defender_score" in scoring_df.columns:
    display(
        scoring_df[
            ["player_name", "utility_defender_score"]
        ].head(10).round(3)
    )
else:
    print("Original composite score column not found.")


Adjusted normalized columns:
['oaa_per_1000_innings', 'versatility_score', 'meaningful_positions', 'demanding_position_share', 'total_innings']

Adjusted normalized preview:


,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share,total_innings
0,0.754,0.704,0.0,1.000,1.000
1,0.572,0.818,0.0,0.308,0.970
2,0.466,0.800,0.0,1.000,0.967
3,0.439,0.261,0.0,1.000,0.963
4,0.171,0.839,0.0,0.681,0.960
5,0.627,0.522,0.5,0.499,0.952
6,0.709,0.756,0.0,1.000,0.948
7,0.385,0.999,0.0,1.000,0.947
8,0.412,0.413,0.0,0.277,0.939
9,0.138,0.869,0.5,0.652,0.925



Original ranking columns:
['player_id', 'player_name', 'total_outs', 'positions_played', 'total_innings', 'qualifying_positions', 'secondary_position', 'secondary_innings', 'secondary_share', 'last_name, first_name', 'display_team_name', 'year', 'primary_pos_formatted', 'fielding_runs_prevented', 'outs_above_average', 'outs_above_average_infront', 'outs_above_average_lateral_toward3bline', 'outs_above_average_lateral_toward1bline', 'outs_above_average_behind', 'outs_above_average_rhh', 'outs_above_average_lhh', 'actual_success_rate_formatted', 'adj_estimated_success_rate_formatted', 'diff_success_rate_formatted', '_merge', 'oaa_per_1000_innings', 'versatility_score', 'meaningful_positions', 'demanding_position_share']

Original composite score preview:


,player_name,utility_defender_score
0,Nico Hoerner,64.155
1,Fernando Tatis Jr.,53.220
2,Brooks Lee,55.716
3,Bo Bichette,41.240
4,Alec Bohm,42.477
5,Mauricio Dubón,57.222
6,Bryson Stott,63.593
7,Kevin McGonigle,57.580
8,Sal Stewart,36.645
9,Luke Keaschall,48.957


In [136]:
# Step 38: Locate composite scoring variables

print("Adjusted normalized columns:")
print(adjusted_normalized.columns.tolist())

print("\nRanked dataframe columns:")
for col in ranked_df.columns:
    print(col)

print("\nFinal rankings columns:")
for col in final_rankings.columns:
    print(col)

print("\nPotential score variables:")
for name, value in globals().copy().items():
    if any(word in name.lower() for word in
           ["score", "composite", "weight", "rank"]):
        if not name.startswith("_"):
            print(name, type(value).__name__)

Adjusted normalized columns:
['oaa_per_1000_innings', 'versatility_score', 'meaningful_positions', 'demanding_position_share', 'total_innings']

Ranked dataframe columns:
player_id
player_name
total_outs
positions_played
total_innings
qualifying_positions
secondary_position
secondary_innings
secondary_share
last_name, first_name
display_team_name
year
primary_pos_formatted
fielding_runs_prevented
outs_above_average
outs_above_average_infront
outs_above_average_lateral_toward3bline
outs_above_average_lateral_toward1bline
outs_above_average_behind
outs_above_average_rhh
outs_above_average_lhh
actual_success_rate_formatted
adj_estimated_success_rate_formatted
diff_success_rate_formatted
_merge
oaa_per_1000_innings
versatility_score
meaningful_positions
demanding_position_share

Final rankings columns:
player_id
player_name
total_outs
positions_played
total_innings
qualifying_positions
secondary_position
secondary_innings
secondary_share
last_name, first_name
display_team_name
year
primary

In [137]:
# Step 39: Confirm final model weights
print("Available scoring scenarios:")
for scenario, weights in scenario_weights.items():
    print(f"\n{scenario}:")
    print(weights)

print("\nAdjusted scoring columns:")
print(adjusted_normalized.columns.tolist())

print("\nAdjusted scoring preview:")
display(adjusted_normalized.head(10).round(3))

Available scoring scenarios:

oaa_per_1000_innings:
0.3

versatility_score:
0.35

meaningful_positions:
0.2

demanding_position_share:
0.1

total_innings:
0.05

Adjusted scoring columns:
['oaa_per_1000_innings', 'versatility_score', 'meaningful_positions', 'demanding_position_share', 'total_innings']

Adjusted scoring preview:


,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share,total_innings
0,0.754,0.704,0.0,1.000,1.000
1,0.572,0.818,0.0,0.308,0.970
2,0.466,0.800,0.0,1.000,0.967
3,0.439,0.261,0.0,1.000,0.963
4,0.171,0.839,0.0,0.681,0.960
5,0.627,0.522,0.5,0.499,0.952
6,0.709,0.756,0.0,1.000,0.948
7,0.385,0.999,0.0,1.000,0.947
8,0.412,0.413,0.0,0.277,0.939
9,0.138,0.869,0.5,0.652,0.925


In [138]:
# Step 40: Calculate final adjusted utility defender rankings

# Define the confirmed model weights
final_weights = {
    "oaa_per_1000_innings": 0.30,
    "versatility_score": 0.35,
    "meaningful_positions": 0.20,
    "demanding_position_share": 0.10,
    "total_innings": 0.05
}

# Confirm scoring columns and weights match
assert set(final_weights) == set(adjusted_normalized.columns)
assert abs(sum(final_weights.values()) - 1.0) < 1e-9

# Align normalized metrics with the original player records
final_model = ranked_df.copy()

assert final_model.index.equals(adjusted_normalized.index)

# Calculate weighted composite score (0-100)
final_model["adjusted_composite_score"] = (
    sum(
        adjusted_normalized[col] * weight
        for col, weight in final_weights.items()
    ) * 100
)

# Sort players by final adjusted score
final_model = final_model.sort_values(
    "adjusted_composite_score",
    ascending=False
).reset_index(drop=True)

# Assign final rankings
final_model["final_rank"] = range(1, len(final_model) + 1)

# Display final leaderboard
display(
    final_model[
        [
            "final_rank",
            "player_name",
            "adjusted_composite_score",
            "total_innings",
            "outs_above_average",
            "versatility_score",
            "meaningful_positions"
        ]
    ].head(20).round(3)
)

print("Total ranked players:", len(final_model))

,final_rank,player_name,adjusted_composite_score,total_innings,outs_above_average,versatility_score,meaningful_positions
0,1,Ezequiel Duran,76.934,1282.333,17.0,0.842,3
1,2,Myles Straw,67.809,760.667,4.0,0.971,3
2,3,Jorbit Vivas,65.513,747.333,4.0,1.000,2
3,4,Sung-Mun Song,63.635,389.333,4.0,0.966,2
4,5,Steven Kwan,63.475,1263.667,6.0,1.000,2
5,6,Brice Matthews,63.192,531.000,6.0,0.787,3
6,7,Bryson Stott,62.449,1321.000,9.0,0.853,2
7,8,Nico Hoerner,62.264,1372.000,11.0,0.822,2
8,9,Jake Mangum,61.410,986.333,-1.0,0.921,3
9,10,Kevin McGonigle,61.258,1319.667,-3.0,1.000,2


Total ranked players: 89


In [139]:
# Step 41: Export final utility defender rankings

from pathlib import Path

# Define export location
output_path = Path("../data/2026_utility_defender_rankings.csv")

# Select final evaluation columns
export_columns = [
    "final_rank",
    "player_id",
    "player_name",
    "adjusted_composite_score",
    "total_innings",
    "outs_above_average",
    "oaa_per_1000_innings",
    "versatility_score",
    "meaningful_positions",
    "demanding_position_share"
]

# Create export dataset
final_export = final_model[export_columns].copy()

# Save correctly ranked CSV
final_export.to_csv(output_path, index=False)

# Verify the export
verified = pd.read_csv(output_path)

assert len(verified) == 89
assert verified["final_rank"].tolist() == list(range(1, 90))
assert verified["adjusted_composite_score"].is_monotonic_decreasing

print("Final rankings successfully exported!")
print("File:", output_path)
print("Players ranked:", len(verified))

display(verified.head(10).round(3))

Final rankings successfully exported!
File: ../data/2026_utility_defender_rankings.csv
Players ranked: 89


,final_rank,player_id,player_name,adjusted_composite_score,total_innings,outs_above_average,oaa_per_1000_innings,versatility_score,meaningful_positions,demanding_position_share
0,1,677649,Ezequiel Duran,76.934,1282.333,17.0,13.257,0.842,3,0.857
1,2,664702,Myles Straw,67.809,760.667,4.0,5.259,0.971,3,0.397
2,3,678391,Jorbit Vivas,65.513,747.333,4.0,5.352,1.000,2,1.000
3,4,823550,Sung-Mun Song,63.635,389.333,4.0,10.274,0.966,2,1.000
4,5,680757,Steven Kwan,63.475,1263.667,6.0,4.748,1.000,2,0.504
5,6,694728,Brice Matthews,63.192,531.000,6.0,11.299,0.787,3,0.731
6,7,681082,Bryson Stott,62.449,1321.000,9.0,6.813,0.853,2,1.000
7,8,663538,Nico Hoerner,62.264,1372.000,11.0,8.017,0.822,2,1.000
8,9,663968,Jake Mangum,61.410,986.333,-1.0,-1.014,0.921,3,0.499
9,10,805808,Kevin McGonigle,61.258,1319.667,-3.0,-2.273,1.000,2,1.000
